# Notebook 05 — Export to server format

This notebook converts the enriched Lyon data (from notebooks 04 and 05) into the
JSON schema expected by `connoisseur/server.py`.

The server expects:
- `data/lyon-restaurants.json` — restaurants with keys: name, location, type, 
  food_style, rating, price_range, signatures, vibe, environment, shortcomings, itemId
- `data/lyon-reviews.json` — reviews with keys: reviewId, userId, itemId, title, 
  text, date, rating, language, etc.
- `data/lyon-culinary-map.txt` — free-text guide (used by the `recommend_by_vibe` 
  second-pass search)

The legacy California files are renamed with `_legacy_california_` prefix as backup.

In [2]:
"""Notebook 05 — Export enriched Lyon data to server-compatible JSON."""

import json
import shutil
from pathlib import Path

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_DIR = PROJECT_ROOT / "data"

# Inputs (from previous notebooks)
ENRICHED_RESTAURANTS = DATA_PROCESSED / "lyon_restaurants_with_vibes.json"
SYNTHETIC_REVIEWS = DATA_PROCESSED / "lyon_reviews.json"

# Outputs (server-compatible)
OUTPUT_RESTAURANTS = DATA_DIR / "lyon-restaurants.json"
OUTPUT_REVIEWS = DATA_DIR / "lyon-reviews.json"
OUTPUT_CULINARY_MAP = DATA_DIR / "lyon-culinary-map.txt"

print(f"Reading from: {DATA_PROCESSED}")
print(f"Writing to:   {DATA_DIR}")

Reading from: C:\Users\busar\Desktop\Projets\connoisseur\data\processed
Writing to:   C:\Users\busar\Desktop\Projets\connoisseur\data


In [3]:
# Backup the original California files with a clear prefix
legacy_renames = {
    "structured-restaurant-data.json": "_legacy_california_restaurants.json",
    "augmented-user-review.json": "_legacy_california_reviews.json",
    "California-Culinary-Map.txt": "_legacy_california_culinary_map.txt",
}

for old_name, new_name in legacy_renames.items():
    old_path = DATA_DIR / old_name
    new_path = DATA_DIR / new_name
    
    if old_path.exists():
        if new_path.exists():
            print(f"⏭️  {new_name} already exists, skipping")
        else:
            old_path.rename(new_path)
            print(f"✅ {old_name} → {new_name}")
    else:
        print(f"⚠️  {old_name} not found (already moved?)")

✅ structured-restaurant-data.json → _legacy_california_restaurants.json
✅ augmented-user-review.json → _legacy_california_reviews.json
✅ California-Culinary-Map.txt → _legacy_california_culinary_map.txt


In [4]:
# Load the enriched restaurants
with open(ENRICHED_RESTAURANTS, "r", encoding="utf-8") as f:
    enriched = json.load(f)

# Load the synthetic reviews  
with open(SYNTHETIC_REVIEWS, "r", encoding="utf-8") as f:
    reviews_raw = json.load(f)

print(f"Loaded {len(enriched)} enriched restaurants")
print(f"Loaded {len(reviews_raw)} synthetic reviews")
print(f"\nSample restaurant fields: {list(enriched[0].keys())}")
print(f"\nSample review fields: {list(reviews_raw[0].keys())}")

Loaded 148 enriched restaurants
Loaded 296 synthetic reviews

Sample restaurant fields: ['restaurant_link', 'restaurant_name', 'primary_cuisine', 'cuisines', 'price_level', 'avg_rating', 'total_reviews_count', 'address', 'vibes', 'description', 'signatures', 'shortcomings']

Sample review fields: ['reviewId', 'userId', 'itemId', 'title', 'text', 'date', 'rating', 'language', 'lean']


In [5]:
def map_restaurant(r: dict, idx: int) -> dict:
    """
    Map an enriched TripAdvisor restaurant to the server's expected format.
    
    The server expects keys: name, location, type, food_style, rating, price_range,
    signatures, vibe, environment, shortcomings, itemId
    """
    # Extract a short location string from the full address
    # e.g. "12 rue du Boeuf, 69005 Lyon, France" → "12 rue du Boeuf, 69005 Lyon"
    address = r.get("address", "")
    location = address.replace(", France", "").strip() if address else "Lyon"
    
    # The description from Claude becomes the "environment"
    environment = r.get("description", "")
    
    # The vibes list maps directly to "vibe"
    vibes = r.get("vibes", [])
    
    return {
        "name": r["restaurant_name"],
        "location": location,
        "type": r.get("primary_cuisine", "Restaurant"),
        "food_style": r.get("cuisines", r.get("primary_cuisine", "")),
        "rating": float(r.get("avg_rating", 0)) if r.get("avg_rating") else 0.0,
        "price_range": r.get("price_level", ""),
        "signatures": r.get("signatures", []),
        "vibe": vibes,
        "environment": environment,
        "shortcomings": r.get("shortcomings", []),
        "itemId": 1000000 + idx,
    }


# Apply the mapping
restaurants_for_server = [map_restaurant(r, idx) for idx, r in enumerate(enriched)]

# Verify
print(f"Mapped {len(restaurants_for_server)} restaurants")
print(f"\nFirst restaurant in new format:")
print(json.dumps(restaurants_for_server[0], indent=2, ensure_ascii=False))

Mapped 148 restaurants

First restaurant in new format:
{
  "name": "Lyon-Dakar",
  "location": "227 rue de Crequi, 69003 Lyon France",
  "type": "African",
  "food_style": "African",
  "rating": 4.0,
  "price_range": "€€-€€€",
  "signatures": [
    "Thiéboudienne (Senegalese fish and rice)",
    "Grilled lamb with sauce d'arachide",
    "Mafé (groundnut stew)"
  ],
  "vibe": [
    "convivial",
    "casual",
    "vibrant",
    "unpretentious",
    "neighbourhood-focused"
  ],
  "environment": "Lyon-Dakar brings authentic West African cooking to the Croix-Rousse neighbourhood, moving beyond the city's traditional Lyonnais repertoire with grilled meats, rich stews, and regional specialities prepared without nostalgia or irony. The restaurant occupies a straightforward dining space where the food speaks louder than decor, drawing a steady crowd of locals and curious eaters seeking genuine flavour over aesthetics.",
  "shortcomings": [
    "Service can be slow during peak hours, reflecting

In [6]:
def map_review(r: dict) -> dict:
    """
    Map a synthetic review to the server's expected format.
    
    The server expects keys: reviewId, userId, itemId, title, text, date, rating,
    language, images, image_captions
    """
    return {
        "reviewId": r["reviewId"],
        "userId": r.get("userId", "USER_ANONYMOUS"),
        "itemId": r["itemId"],
        "title": r.get("title", ""),
        "text": r.get("text", ""),
        "date": r.get("date", "2025-01-01"),
        "rating": float(r.get("rating", 4.0)),
        "language": r.get("language", "en"),
        "images": "[]",  # we don't have synthetic images
        "image_captions": [],  # idem
    }


reviews_for_server = [map_review(r) for r in reviews_raw]

print(f"Mapped {len(reviews_for_server)} reviews")
print(f"\nFirst review in new format:")
print(json.dumps(reviews_for_server[0], indent=2, ensure_ascii=False))

Mapped 296 reviews

First review in new format:
{
  "reviewId": 900000000,
  "userId": "USER_LYON_FOOD_ADVENTURER",
  "itemId": 1000000,
  "title": "Authentic West African food, worth the wait",
  "text": "Took my partner here on a Friday night and we were genuinely impressed. The thiéboudienne was excellent – properly seasoned rice, fresh fish, and that perfect balance of spice. We also tried the mafé, which had real depth and wasn't overly heavy despite the groundnut sauce. The grilled lamb was tender and the sauce d'arachide complemented it nicely. Service was friendly and casual, though I'll be honest – we waited quite a while between courses, even with just a few tables occupied. But honestly, it felt like they were taking time to get the food right rather than rushing us out. The wine list is pretty basic, mostly affordable reds, so don't expect creative pairings. That said, the prices are fair for the quality and portion sizes. The neighbourhood vibe is exactly what Lyon needed 

In [7]:
def restaurant_to_paragraph(r: dict) -> str:
    """Format one restaurant as a free-text paragraph for the culinary map."""
    name = r["name"]
    cuisine = r["type"]
    location = r["location"]
    rating = r["rating"]
    price = r["price_range"]
    vibes = ", ".join(r.get("vibe", []))
    environment = r.get("environment", "")
    signatures = ", ".join(r.get("signatures", []))
    
    lines = [
        f"## {name}",
        f"Cuisine: {cuisine} · Location: {location} · Rating: {rating}/5 · Price: {price}",
        f"Atmosphere: {vibes}.",
        environment,
        f"Notable dishes: {signatures}." if signatures else "",
    ]
    return "\n".join(l for l in lines if l)


# Build the full map
header = """# Lyon Culinary Map

A curated guide to ~40 quality restaurants in Lyon, France.
Generated from TripAdvisor data enriched with AI-extracted vibes and descriptions.
"""

paragraphs = [restaurant_to_paragraph(r) for r in restaurants_for_server]
culinary_map_text = header + "\n\n" + "\n\n".join(paragraphs)

# Preview
print(culinary_map_text[:800] + "\n...(truncated)")

# Lyon Culinary Map

A curated guide to ~40 quality restaurants in Lyon, France.
Generated from TripAdvisor data enriched with AI-extracted vibes and descriptions.


## Lyon-Dakar
Cuisine: African · Location: 227 rue de Crequi, 69003 Lyon France · Rating: 4.0/5 · Price: €€-€€€
Atmosphere: convivial, casual, vibrant, unpretentious, neighbourhood-focused.
Lyon-Dakar brings authentic West African cooking to the Croix-Rousse neighbourhood, moving beyond the city's traditional Lyonnais repertoire with grilled meats, rich stews, and regional specialities prepared without nostalgia or irony. The restaurant occupies a straightforward dining space where the food speaks louder than decor, drawing a steady crowd of locals and curious eaters seeking genuine flavour over aesthetics.
Notable dishes: Thi
...(truncated)


In [9]:
# Write restaurants JSON
with open(OUTPUT_RESTAURANTS, "w", encoding="utf-8") as f:
    json.dump(restaurants_for_server, f, indent=2, ensure_ascii=False)
print(f"✅ Wrote {OUTPUT_RESTAURANTS.name} ({OUTPUT_RESTAURANTS.stat().st_size / 1024:.1f} KB)")

# Write reviews JSON
with open(OUTPUT_REVIEWS, "w", encoding="utf-8") as f:
    json.dump(reviews_for_server, f, indent=2, ensure_ascii=False)
print(f"✅ Wrote {OUTPUT_REVIEWS.name} ({OUTPUT_REVIEWS.stat().st_size / 1024:.1f} KB)")

# Write culinary map text
OUTPUT_CULINARY_MAP.write_text(culinary_map_text, encoding="utf-8")
print(f"✅ Wrote {OUTPUT_CULINARY_MAP.name} ({OUTPUT_CULINARY_MAP.stat().st_size / 1024:.1f} KB)")

✅ Wrote lyon-restaurants.json (176.3 KB)
✅ Wrote lyon-reviews.json (344.6 KB)
✅ Wrote lyon-culinary-map.txt (101.0 KB)


In [10]:
# Re-read the files to make sure they're valid JSON
import json as _json

with open(OUTPUT_RESTAURANTS, "r", encoding="utf-8") as f:
    check_restaurants = _json.load(f)
print(f"✅ {OUTPUT_RESTAURANTS.name}: {len(check_restaurants)} restaurants, valid JSON")

with open(OUTPUT_REVIEWS, "r", encoding="utf-8") as f:
    check_reviews = _json.load(f)
print(f"✅ {OUTPUT_REVIEWS.name}: {len(check_reviews)} reviews, valid JSON")

# Check that all reviews have a matching restaurant via itemId
restaurant_ids = {r["itemId"] for r in check_restaurants}
review_ids = {r["itemId"] for r in check_reviews}
orphan_reviews = review_ids - restaurant_ids

if orphan_reviews:
    print(f"⚠️  {len(orphan_reviews)} reviews have no matching restaurant: {orphan_reviews}")
else:
    print(f"✅ All review itemIds match a restaurant ({len(review_ids)} unique IDs)")

# Quick check that key fields are populated
sample = check_restaurants[0]
required_keys = ["name", "location", "type", "rating", "price_range", "signatures", "vibe", "environment", "shortcomings", "itemId"]
missing = [k for k in required_keys if k not in sample]
if missing:
    print(f"⚠️  Missing keys in restaurants: {missing}")
else:
    print(f"✅ All required keys present in restaurants")

✅ lyon-restaurants.json: 148 restaurants, valid JSON
✅ lyon-reviews.json: 296 reviews, valid JSON
✅ All review itemIds match a restaurant (148 unique IDs)
✅ All required keys present in restaurants
